<img src="https://s8.hostingkartinok.com/uploads/images/2018/08/308b49fcfbc619d629fe4604bceb67ac.jpg" width=500, height=450>
<h3 style="text-align: center;"><b>Физтех-Школа Прикладной математики и информатики (ФПМИ) МФТИ</b></h3>

---

***Some parts of the notebook are almost the copy of [ mmta-team course](https://github.com/mmta-team/mmta_fall_2020). Special thanks to mmta-team for making them publicly available. [Original notebook](https://github.com/mmta-team/mmta_fall_2020/blob/master/tasks/01_word_embeddings/task_word_embeddings.ipynb).***

<b> Прочитайте семинар, пожалуйста, для успешного выполнения домашнего задания. В конце ноутка напишите свой вывод. Работа без вывода оценивается ниже.

## Задача поиска схожих по смыслу предложений

Мы будем ранжировать вопросы [StackOverflow](https://stackoverflow.com) на основе семантического векторного представления

До этого в курсе не было речи про задачу ранжировния, поэтому введем математическую формулировку

## Задача ранжирования(Learning to Rank)

* $X$ - множество объектов
* $X^l = \{x_1, x_2, ..., x_l\}$ - обучающая выборка
<br>На обучающей выборке задан порядок между некоторыми элементами, то есть нам известно, что некий объект выборки более релевантный для нас, чем другой:
* $i \prec j$ - порядок пары индексов объектов на выборке $X^l$ c индексами $i$ и $j$
### Задача:
построить ранжирующую функцию $a$ : $X \rightarrow R$ такую, что
$$i \prec j \Rightarrow a(x_i) < a(x_j)$$

<img src="https://d25skit2l41vkl.cloudfront.net/wp-content/uploads/2016/12/Featured-Image.jpg" width=500, height=450>

### Embeddings

Будем использовать предобученные векторные представления слов на постах Stack Overflow.<br>
[A word2vec model trained on Stack Overflow posts](https://github.com/vefstathiou/SO_word2vec)

In [ ]:
%pip install -U "numpy==1.26.4" "scipy==1.12.0" "gensim==4.3.3" "smart_open>=6.4.0"


In [ ]:
import gensim, numpy as np, scipy
print("gensim", gensim.__version__, "| numpy", np.__version__, "| scipy", scipy.__version__)


gensim 4.3.3 | numpy 1.26.4 | scipy 1.12.0


In [ ]:
!wget https://zenodo.org/record/1199620/files/SO_vectors_200.bin?download=1

--2025-09-25 13:49:26--  https://zenodo.org/record/1199620/files/SO_vectors_200.bin?download=1
Resolving zenodo.org (zenodo.org)... 188.185.45.92, 188.185.48.194, 188.185.43.25, ...
Connecting to zenodo.org (zenodo.org)|188.185.45.92|:443... connected.
HTTP request sent, awaiting response... 301 MOVED PERMANENTLY
Location: /records/1199620/files/SO_vectors_200.bin [following]
--2025-09-25 13:49:26--  https://zenodo.org/records/1199620/files/SO_vectors_200.bin
Reusing existing connection to zenodo.org:443.
HTTP request sent, awaiting response... 200 OK
Length: 1453905423 (1.4G) [application/octet-stream]
Saving to: ‘SO_vectors_200.bin?download=1’

SO_vectors_200.bin? 100%[===================>]   1.35G  4.62MB/s    in 6m 32s  

2025-09-25 13:55:59 (3.54 MB/s) - ‘SO_vectors_200.bin?download=1’ saved [1453905423/1453905423]



In [ ]:
from gensim.models.keyedvectors import KeyedVectors
wv_embeddings = KeyedVectors.load_word2vec_format("SO_vectors_200.bin?download=1", binary=True)

#### Как пользоваться этими векторами?

Посмотрим на примере одного слова, что из себя представляет embedding

In [ ]:
word = 'dog'
if word in wv_embeddings:
    print(wv_embeddings[word].dtype, wv_embeddings[word].shape)

float32 (200,)


In [ ]:
print(f"Num of words: {len(wv_embeddings.index_to_key)}")

Num of words: 1787145


Найдем наиболее близкие слова к слову `dog`:

#### ***Вопрос 1:***
* Входит ли слово `cat` в топ-5 близких слов к слову `dog`? Какое место оно занимает?


In [ ]:
wv_embeddings.has_index_for("cat")

True

In [ ]:
wv_embeddings.most_similar("dog", topn=5)


[('animal', 0.8564180135726929),
 ('dogs', 0.7880866527557373),
 ('mammal', 0.7623804211616516),
 ('cats', 0.7621253728866577),
 ('animals', 0.760793924331665)]

***Ваш ответ:*** 'cat' не входит в топ-5 ближайших слов к 'dog'.

А вот 'cats' входит :) Также как и animal, dogs, mammal и animals.

### Векторные представления текста

Перейдем от векторных представлений отдельных слов к векторным представлениям вопросов, как к **среднему** векторов всех слов в вопросе. Если для какого-то слова нет предобученного вектора, то его нужно пропустить. Если вопрос не содержит ни одного известного слова, то нужно вернуть нулевой вектор.

Так как имеем дело с постами StackOverflow, скипать всю на свете пунктуацию - плохая идея:

In [ ]:
wv_embeddings.has_index_for("c#")

True

In [ ]:
wv_embeddings.has_index_for("c++")

True

Это токенайзер из задания

In [ ]:
import numpy as np
import re
class MyTokenizer:
    def __init__(self):
        pass
    def tokenize(self, text):
        return re.findall(r'\w+', text)

tokenizer = MyTokenizer()

А это токенайзер, не срезающий пунктуацию и спецсимволы, и приводящий текст к нижнему регистру. Дальше я буду сравнивать их работу

In [ ]:
import re
class ReallyMyTokenizer:
    def __init__(self, lower=True):
        self.pat = re.compile(r"[A-Za-z0-9_+#.-]+")
        self.lower = lower
    def tokenize(self, text: str):
        toks = self.pat.findall(text)
        if self.lower:
          return [t.lower() for t in toks]
        else:
          return toks

my_tokenizer = ReallyMyTokenizer()


In [ ]:
def question_to_vec(question, embeddings, tokenizer, dim=200):
    """
        question: строка
        embeddings: наше векторное представление
        dim: размер любого вектора в нашем представлении

        return: векторное представление для вопроса
    """
    tokens = tokenizer.tokenize(question)
    vecs = []
    for token in tokens:
        if token in embeddings:
            vecs.append(embeddings[token])
    if len(vecs) == 0:
        return np.zeros(dim)
    else:
        return np.mean(vecs, axis=0)

Теперь у нас есть метод для создания векторного представления любого предложения.

#### ***Вопрос 2:***

* Какая третья (с индексом 2) компонента вектора предложения `I love neural networks` (округлите до 2 знаков после запятой)?

In [ ]:
question = "I love neural networks"
v1 = question_to_vec(question, wv_embeddings, tokenizer)
print(f"{v1[2]:.2f}")

v2 = question_to_vec(question, wv_embeddings, my_tokenizer)
print(f"{v2[2]:.2f}")


-1.29
-1.29


Ответ: -1.29

### Оценка близости текстов

Представим, что мы используем идеальные векторные представления слов. Тогда косинусное расстояние между дублирующими предложениями должно быть меньше, чем между случайно взятыми предложениями.

Сгенерируем для каждого из $N$ вопросов $R$ случайных отрицательных примеров и примешаем к ним также настоящие дубликаты. Для каждого вопроса будем ранжировать с помощью нашей модели $R + 1$ примеров и смотреть на позицию дубликата. Мы хотим, чтобы дубликат был первым в ранжированном списке.

#### Hits@K
Первой простой метрикой будет количество корректных попаданий для какого-то $K$:
$$ \text{Hits@K} = \frac{1}{N}\sum_{i=1}^N \, [rank\_q_i^{'} \le K],$$
* $\begin{equation*}
[x < 0 ] \equiv
 \begin{cases}
   1, &x < 0\\
   0, &x \geq 0
 \end{cases}
\end{equation*}$ - индикаторная функция
* $q_i$ - $i$-ый вопрос
* $q_i^{'}$ - его дубликат
* $rank\_q_i^{'}$ - позиция дубликата в ранжированном списке ближайших предложений для вопроса $q_i$.

Hits@K  измеряет долю вопросов, для которых правильный ответ попал в топ-K позиций среди отранжированных кандидатов.

#### DCG@K
Второй метрикой будет упрощенная DCG метрика, учитывающая порядок элементов в списке путем домножения релевантности элемента на вес равный обратному логарифму номера позиции::
$$ \text{DCG@K} = \frac{1}{N} \sum_{i=1}^N\frac{1}{\log_2(1+rank\_q_i^{'})}\cdot[rank\_q_i^{'} \le K],$$
С такой метрикой модель штрафуется за большой ранк корректного ответа.

DCG@K  измеряет качество ранжирования, учитывая не только факт наличия правильного ответа в топ-K, но и ***его точную позицию***.

<img src='https://hsto.org/files/1c5/edf/dee/1c5edfdeebce4b71a86bdf986d9f88f2.jpg' width=400, height=200>

#### Пример оценок

Вычислим описанные выше метрики для игрушечного примера.
Пусть
* $N = 1$, $R = 3$
* <font color='green'>"Что такое python?"</font> - вопрос $q_1$
* <font color='red'>"Что такое язык python?"</font> - его дубликат $q_i^{'}$

Пусть модель выдала следующий ранжированный список кандидатов:

1. "Как изучить с++?"
2. <font color='red'>"Что такое язык python?"</font>
3. "Хочу учить Java"
4. "Не понимаю Tensorflow"

$\Rightarrow rank\_q_i^{'} = 2$

Вычислим метрику *Hits@K* для *K = 1, 4*:

- [K = 1] $\text{Hits@1} =  [rank\_q_i^{'} \le 1]$

Проверяем условие $ \text{rank}_{q'_1} \leq 1 $: ***условие неверно***.

Следовательно, $[\text{rank}_{q'_1} \leq 1] = 0$.

- [K = 4] $\text{Hits@4} =  [rank\_q_i^{'} \le 4] = 1$

Проверяем условие $ \text{rank}_{q'_1} \leq 4 $: ***условие верно***.

Вычислим метрику *DCG@K* для *K = 1, 4*:
- [K = 1] $\text{DCG@1} = \frac{1}{\log_2(1+2)}\cdot[2 \le 1] = 0$
- [K = 4] $\text{DCG@4} = \frac{1}{\log_2(1+2)}\cdot[2 \le 4] = \frac{1}{\log_2{3}}$

#### ***Вопрос 3***:
* Вычислите `DCG@10`, если $rank\_q_i^{'} = 9$(округлите до одного знака после запятой)



Вычислим метрику $\text{DCG@K}$ для $K = 10$:

$$\text{DCG@10} = \frac{1}{\log_2(1+9)}\cdot[9 \le 10] = \frac{1}{\log_2{10}},$$


$$\boxed{\text{DCG@10} \approx 0.3}$$

In [ ]:
print(f"DCG@10 = {1 / np.log2(10):.1f}")

DCG@10 = 0.3


#### Более сложный пример оценок

Рассмотрим пример с $ N > 1 $, где $ N = 3 $ (три вопроса) и для каждого вопроса заданы позиции их дубликатов. Вычислим метрики **Hits@K** для разных значений $ K $.

---

- $ N = 3 $: Три вопроса ($ q_1, q_2, q_3 $).
- Для каждого вопроса известна позиция его дубликата ($ \text{rank}_{q'_i} $):
  - $ \text{rank}_{q'_1} = 2 $,
  - $ \text{rank}_{q'_2} = 5 $,
  - $ \text{rank}_{q'_3} = 1 $.

Мы будем вычислять **Hits@K** для $ K = 1, 5 $.

---

**Для $ K = 1 $:**

Подставим значения:
$$
\text{Hits@1} = \frac{1}{3} \cdot \left( [\text{rank}_{q'_1} \leq 1] + [\text{rank}_{q'_2} \leq 1] + [\text{rank}_{q'_3} \leq 1] \right).
$$

Проверяем условие $ \text{rank}_{q'_i} \leq 1 $ для каждого вопроса:
- $ \text{rank}_{q'_1} = 2 $ → $ 2 \not\leq 1 $ → $ 0 $,
- $ \text{rank}_{q'_2} = 5 $ → $ 5 \not\leq 1 $ → $ 0 $,
- $ \text{rank}_{q'_3} = 1 $ → $ 1 \leq 1 $ → $ 1 $.

Сумма:
$$
\text{Hits@1} = \frac{1}{3} \cdot (0 + 0 + 1) = \frac{1}{3}.
$$

$$
\boxed{\text{Hits@1} = \frac{1}{3}}.
$$

---

**Для $ K = 5 $:**

Подставим значения:
$$
\text{Hits@5} = \frac{1}{3} \cdot \left( [\text{rank}_{q'_1} \leq 5] + [\text{rank}_{q'_2} \leq 5] + [\text{rank}_{q'_3} \leq 5] \right).
$$

Проверяем условие $ \text{rank}_{q'_i} \leq 5 $ для каждого вопроса:
- $ \text{rank}_{q'_1} = 2 $ → $ 2 \leq 5 $ → $ 1 $,
- $ \text{rank}_{q'_2} = 5 $ → $ 5 \leq 5 $ → $ 1 $,
- $ \text{rank}_{q'_3} = 1 $ → $ 1 \leq 5 $ → $ 1 $.

Сумма:
$$
\text{Hits@5} = \frac{1}{3} \cdot (1 + 1 + 1) = 1.
$$

$$
\boxed{\text{Hits@5} = 1}.
$$

Теперь вычислим метрику **DCG@K** для того же примера, где $ N = 3 $ (три вопроса), и для каждого вопроса известна позиция его дубликата ($ \text{rank}_{q'_i} $):

- $ \text{rank}_{q'_1} = 2 $,
- $ \text{rank}_{q'_2} = 5 $,
- $ \text{rank}_{q'_3} = 1 $.

Мы будем вычислять **DCG@K** для $ K = 1, 5 $.

---
**Для $ K = 1 $:**
Подставим значения:
$$
\text{DCG@1} = \frac{1}{3} \cdot \left( \frac{1}{\log_2(1 + \text{rank}_{q'_1})} \cdot [\text{rank}_{q'_1} \leq 1] + \frac{1}{\log_2(1 + \text{rank}_{q'_2})} \cdot [\text{rank}_{q'_2} \leq 1] + \frac{1}{\log_2(1 + \text{rank}_{q'_3})} \cdot [\text{rank}_{q'_3} \leq 1] \right).
$$

Проверяем условие $ \text{rank}_{q'_i} \leq 1 $ для каждого вопроса:
- $ \text{rank}_{q'_1} = 2 $ → $ 2 \not\leq 1 $ → $ 0 $,
- $ \text{rank}_{q'_2} = 5 $ → $ 5 \not\leq 1 $ → $ 0 $,
- $ \text{rank}_{q'_3} = 1 $ → $ 1 \leq 1 $ → $ 1 $.

Сумма:
$$
\text{DCG@1} = \frac{1}{3} \cdot (0 + 0 + 1) = \frac{1}{3}.
$$
$$
\boxed{\text{DCG@1} = \frac{1}{3}}.
$$

---


**Для $ K = 5 $:**
Подставим значения:
$$
\text{DCG@5} = \frac{1}{3} \cdot \left( \frac{1}{\log_2(1 + \text{rank}_{q'_1})} \cdot [\text{rank}_{q'_1} \leq 5] + \frac{1}{\log_2(1 + \text{rank}_{q'_2})} \cdot [\text{rank}_{q'_2} \leq 5] + \frac{1}{\log_2(1 + \text{rank}_{q'_3})} \cdot [\text{rank}_{q'_3} \leq 5] \right).
$$

Проверяем условие $ \text{rank}_{q'_i} \leq 5 $ для каждого вопроса:
- $ \text{rank}_{q'_1} = 2 $ → $ 2 \leq 5 $ → $ 1 $,
- $ \text{rank}_{q'_2} = 5 $ → $ 5 \leq 5 $ → $ 1 $,
- $ \text{rank}_{q'_3} = 1 $ → $ 1 \leq 5 $ → $ 1 $.

Сумма:
$$
\text{DCG@5} = \frac{1}{3} \cdot (0.631 + 0.387 + 1) = \frac{1}{3} \cdot 2.018 \approx 0.673.
$$

$$
\boxed{\text{DCG@5} \approx 0.673}.
$$

#### ***Вопрос 4:***
* Найдите максимум `Hits@47 - DCG@1`?

$$ \text{Hits@47} = \frac{1}{N}\sum_{i=1}^N \, [rank\_q_i^{'} \le 47]\le 1$$

$$ \text{DCG@1} = \frac{1}{N} \sum_{i=1}^N\frac{1}{\log_2(1+rank\_q_i^{'})}\cdot[rank\_q_i^{'} \le 1]\ge0$$

Тогда $$\boxed{max(\text{Hits@47}-\text{DCG@1}) = 1}.$$


Это возможно, например, если у нас есть 10 вопросов ($ q_1, q_2, q_3, ..., q_{10} $), и для каждого вопроса позиция его дубликата $ \text{rank}_{q'_i} = i+1$:

 $ \text{rank}_{q'_1} = 2 $, $ \text{rank}_{q'_2} = 3 $, ..., $ \text{rank}_{q'_{10}} = 11 $.

  Тогда $$ \text{Hits@47} = \frac{1}{10}\sum_{i=1}^{10} \, [rank\_q_i^{'} \le 47]=\frac{1}{10}\sum_{i=1}^{10} \, 1=1,$$
  $$ \text{DCG@1} = \frac{1}{10}\sum_{i=1}^{10} \, [rank\_q_i^{'} \le 1]= \frac{1}{10}\sum_{i=1}^{10} \, 0=0,$$


  $$\text{Hits@47}-\text{DCG@1} = 1-0=1.$$

### HITS\_COUNT и DCG\_SCORE

Каждая функция имеет два аргумента: $dup\_ranks$ и $k$.

$dup\_ranks$ является списком, который содержит рейтинги дубликатов (их позиции в ранжированном списке).

К примеру для <font color='red'>"Что такое язык python?"</font> $dup\_ranks = [2]$.

In [ ]:
def hits_count(dup_ranks, k):
    """
        dup_ranks: list индексов дубликатов
        k: пороговое значение для ранга
        result: вернуть Hits@k
    """
    # Подсчитываем количество дубликатов, чей ранг <= k
    hits_value=np.mean((np.array(dup_ranks) <= k).astype(float))
    return hits_value

In [ ]:
dup_ranks = [2]

k = 1
hits_value = hits_count(dup_ranks, k)
print(f"Hits@1 = {hits_value}")

k = 4
hits_value = hits_count(dup_ranks, k)
print(f"Hits@4 = {hits_value}")

Hits@1 = 0.0
Hits@4 = 1.0


In [ ]:
import math

def dcg_score(dup_ranks, k):
    """
        dup_ranks: list индексов дубликатов
        k: пороговое значение для ранга
        result: вернуть DCG@k
    """
    # Вычисляем сумму для всех релевантных дубликатов
    dup_ranks = np.array(dup_ranks)
    sum = np.sum((1/(np.log2(1+dup_ranks)))*(dup_ranks<=k))
    # Делим на общее количество вопросов
    dcg_value=(sum/dup_ranks.shape[0]).astype(float)
    return dcg_value

In [ ]:
# Пример списка позиций дубликатов
dup_ranks = [2]

# Вычисляем DCG@1
dcg_value = dcg_score(dup_ranks, k=1)
print(f"DCG@1 = {dcg_value:.3f}")

# Вычисляем DCG@4
dcg_value = dcg_score(dup_ranks, k=4)
print(f"DCG@10 = {dcg_value:.3f}")

DCG@1 = 0.000
DCG@10 = 0.631


Протестируем функции. Пусть $N = 1$, то есть один эксперимент. Будем искать копию вопроса и оценивать метрики.

In [ ]:
import pandas as pd

In [ ]:
copy_answers = ["How does the catch keyword determine the type of exception that was thrown",]

# наши кандидаты
candidates_ranking = [["How Can I Make These Links Rotate in PHP",
                       "How does the catch keyword determine the type of exception that was thrown",
                       "NSLog array description not memory address",
                       "PECL_HTTP not recognised php ubuntu"],]

# dup_ranks — позиции наших копий, так как эксперимент один, то этот массив длины 1
dup_ranks = [candidates_ranking[0].index(copy_answers[0]) + 1]

# вычисляем метрику для разных k
print('Ваш ответ HIT:', [hits_count(dup_ranks, k) for k in range(1, 5)])
print('Ваш ответ DCG:', [round(dcg_score(dup_ranks, k), 5) for k in range(1, 5)])

Ваш ответ HIT: [0.0, 1.0, 1.0, 1.0]
Ваш ответ DCG: [0.0, 0.63093, 0.63093, 0.63093]


У вас должно получиться

In [ ]:
# correct_answers - метрика для разных k
correct_answers = pd.DataFrame([[0, 1, 1, 1], [0, 1 / (np.log2(3)), 1 / (np.log2(3)), 1 / (np.log2(3))]],
                               index=['HITS', 'DCG'], columns=range(1,5))
correct_answers

,1,2,3,4
HITS,0,1.00000,1.00000,1.00000
DCG,0,0.63093,0.63093,0.63093


### Данные
[arxiv link](https://drive.google.com/file/d/1QqT4D0EoqJTy7v9VrNCYD-m964XZFR7_/edit)

`train.tsv` - выборка для обучения.<br> В каждой строке через табуляцию записаны: **<вопрос>, <похожий вопрос>**

`validation.tsv` - тестовая выборка.<br> В каждой строке через табуляцию записаны: **<вопрос>, <похожий вопрос>, <отрицательный пример 1>, <отрицательный пример 2>, ...**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!unzip -q "/content/drive/MyDrive/Share/stackoverflow_similar_questions.zip" -d "/content/data"

Считайте данные.

In [ ]:
def read_corpus(filename):
    data = []
    with open(filename, encoding='utf-8') as file:
        for line in file:
            data.append(line.strip().split('\t'))
    return data

Нам понадобиться только файл validation.

In [ ]:
validation_data = read_corpus('/content/data/data/validation.tsv')

Кол-во строк

In [ ]:
len(validation_data)

3760

Размер нескольких первых строк

In [ ]:
for i in range(25):
    print(i + 1, len(validation_data[i]))

1 1001
2 1001
3 1001
4 1001
5 1001
6 1001
7 1001
8 1001
9 1001
10 1001
11 1001
12 1001
13 1001
14 1001
15 1001
16 1001
17 1001
18 1001
19 1001
20 1001
21 1001
22 1001
23 1001
24 1001
25 1001


### Ранжирование без обучения

Реализуйте функцию ранжирования кандидатов на основе косинусного расстояния. Функция должна по списку кандидатов вернуть отсортированный список пар (позиция в исходном списке кандидатов, кандидат). При этом позиция кандидата в полученном списке является его рейтингом (первый - лучший). Например, если исходный список кандидатов был [a, b, c], и самый похожий на исходный вопрос среди них - c, затем a, и в конце b, то функция должна вернуть список **[(2, c), (0, a), (1, b)]**.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
from copy import deepcopy

In [ ]:
def rank_candidates(question, candidates, embeddings, tokenizer, dim=200):
    """
        question: строка
        candidates: массив строк(кандидатов) [a, b, c]
        result: пары (начальная позиция, кандидат) [(2, c), (0, a), (1, b)]
    """
    pairs = []
    q_vec = question_to_vec(question, embeddings, tokenizer, dim)
    for p, candidate in enumerate(candidates):
        c_vec = question_to_vec(candidate, embeddings, tokenizer, dim)
        pairs.append((p, candidate, cosine_similarity([q_vec], [c_vec])[0][0]))
    result = sorted(pairs, key=lambda x: x[2], reverse=True)
    return result

Протестируйте работу функции на примерах ниже. Пусть $N=2$, то есть два эксперимента

In [ ]:
questions = ['converting string to list', 'Sending array via Ajax fails']

candidates = [['Convert Google results object (pure js) to Python object', # первый эксперимент
               'C# create cookie from string and send it',
               'How to use jQuery AJAX for an outside domain?'],

              ['Getting all list items of an unordered list in PHP',      # второй эксперимент
               'WPF- How to update the changes in list item of a list',
               'select2 not displaying search results']]

Токинайзер из задания

In [ ]:
tokenizer = MyTokenizer()

In [ ]:
for question, q_candidates in zip(questions, candidates):
        ranks = rank_candidates(question, q_candidates, wv_embeddings, tokenizer)
        print(ranks)
        print()

[(1, 'C# create cookie from string and send it', 0.546291), (0, 'Convert Google results object (pure js) to Python object', 0.47886285), (2, 'How to use jQuery AJAX for an outside domain?', 0.16188455)]

[(1, 'WPF- How to update the changes in list item of a list', 0.4768912), (0, 'Getting all list items of an unordered list in PHP', 0.45695472), (2, 'select2 not displaying search results', 0.16329032)]



Мой токенайзер

In [ ]:
tokenizer = ReallyMyTokenizer()

In [ ]:
for question, q_candidates in zip(questions, candidates):
        ranks = rank_candidates(question, q_candidates, wv_embeddings, tokenizer)
        print(ranks)
        print()

[(1, 'C# create cookie from string and send it', 0.550269), (0, 'Convert Google results object (pure js) to Python object', 0.5414996), (2, 'How to use jQuery AJAX for an outside domain?', 0.097037084)]

[(0, 'Getting all list items of an unordered list in PHP', 0.4613238), (1, 'WPF- How to update the changes in list item of a list', 0.34710348), (2, 'select2 not displaying search results', 0.32301265)]



Для первого экперимента вы можете полностью сравнить ваши ответы и правильные ответы. Но для второго эксперимента два ответа на кандидаты будут <b>скрыты</b>(*)

In [ ]:
# должно вывести
results = [[(1, 'C# create cookie from string and send it'),
            (0, 'Convert Google results object (pure js) to Python object'),
            (2, 'How to use jQuery AJAX for an outside domain?')],
           [(*, 'Getting all list items of an unordered list in PHP'), #скрыт
            (*, 'select2 not displaying search results'), #скрыт
            (*, 'WPF- How to update the changes in list item of a list')]] #скрыт

Последовательность начальных индексов вы должны получить `для эксперимента 1`  1, 0, 2.

#### ***Вопрос 5:***
* Какую последовательность начальных индексов вы получили `для эксперимента 2`(перечисление без запятой и пробелов, например, `102` для первого эксперимента?


Ответ:


102 на токенайзере из задания, фильтрующем пунктуацию.


012 на собственном токенайзере, не отфильтровывающем пунктуацию.

Теперь мы можем оценить качество нашего метода. Запустите следующие два блока кода для получения результата. Обратите внимание, что вычисление расстояния между векторами занимает некоторое время (примерно 10 минут). Можете взять для validation 1000 примеров.

In [ ]:
tokenizer = MyTokenizer()

In [ ]:
from tqdm.notebook import tqdm

In [ ]:
wv_ranking = []
max_validation_examples = 1000
for i, line in enumerate(tqdm(validation_data)):
    if i == max_validation_examples:
        break
    q, *ex = line
    ranks = rank_candidates(q, ex, wv_embeddings, tokenizer)
    wv_ranking.append([r[0] for r in ranks].index(0) + 1)

  0%|          | 0/3760 [00:00<?, ?it/s]

In [ ]:
for k in tqdm([1, 5, 10, 100, 500, 1000]):
    print("DCG@%4d: %.3f | Hits@%4d: %.3f" % (k, dcg_score(wv_ranking, k), k, hits_count(wv_ranking, k)))

  0%|          | 0/6 [00:00<?, ?it/s]

DCG@   1: 0.285 | Hits@   1: 0.285
DCG@   5: 0.342 | Hits@   5: 0.393
DCG@  10: 0.360 | Hits@  10: 0.449
DCG@ 100: 0.406 | Hits@ 100: 0.679
DCG@ 500: 0.431 | Hits@ 500: 0.879
DCG@1000: 0.444 | Hits@1000: 1.000


In [ ]:
tokenizer = ReallyMyTokenizer()

In [ ]:
wv_ranking = []
max_validation_examples = 1000
for i, line in enumerate(tqdm(validation_data)):
    if i == max_validation_examples:
        break
    q, *ex = line
    ranks = rank_candidates(q, ex, wv_embeddings, tokenizer)
    wv_ranking.append([r[0] for r in ranks].index(0) + 1)

  0%|          | 0/3760 [00:00<?, ?it/s]

In [ ]:
for k in tqdm([1, 5, 10, 100, 500, 1000]):
    print("DCG@%4d: %.3f | Hits@%4d: %.3f" % (k, dcg_score(wv_ranking, k), k, hits_count(wv_ranking, k)))

  0%|          | 0/6 [00:00<?, ?it/s]

DCG@   1: 0.407 | Hits@   1: 0.407
DCG@   5: 0.495 | Hits@   5: 0.575
DCG@  10: 0.516 | Hits@  10: 0.641
DCG@ 100: 0.563 | Hits@ 100: 0.868
DCG@ 500: 0.577 | Hits@ 500: 0.972
DCG@1000: 0.580 | Hits@1000: 1.000


Из формул выше можно понять, что

- $ \text{Hits@K} $ **монотонно неубывающая функция** $ K $, которая стремится к 1 при $ K \to \infty $.

- $ \text{DCG@K} $ **монотонно неубывающая функция** $ K $, но рост замедляется с увеличением $ K $ из-за убывания веса $ \frac{1}{\log_2(1 + \text{rank}_{q'_i})} $.

### Эмбеддинги, обученные на корпусе похожих вопросов

In [ ]:
train_data = read_corpus('/content/data/data/train.tsv')

Улучшите качество модели.<br>Склеим вопросы в пары и обучим на них модель Word2Vec из gensim. Выберите размер window. Объясните свой выбор.

***Рассмотрим подробнее*** данное склеивание.

1. Каждая строка из train_data разбивается на вопрос (question) и список кандидатов.

2. Для каждого кандидата вопрос склеивается с ним в одну строку.

3. Склеенная строка (combined_text) токенизируется, и полученный список токенов добавляется в общий корпус (corpus).

***Пример***

    Вопрос: "What is Python?"
    Кандидаты: ["Python is a programming language", "Java is another language"]
    Склеенные строки:
        "What is Python? Python is a programming language"
        "What is Python? Java is another language"
         
    Токенизированные списки:
        ['what', 'is', 'python', 'python', 'is', 'a', 'programming', 'language']
        ['what', 'is', 'python', 'java', 'is', 'another', 'language']
         
     

In [ ]:
len(train_data)

1000000

In [ ]:
train_data[111258]

['Determine if the device is a smartphone or tablet?',
 'Change imageView params in all cards together']

Эксперимент с токенайзером из ноутбука

In [ ]:
tokenizer = MyTokenizer()

In [ ]:
# Создаем общий корпус текстов
corpus = []
for line in train_data:
    question, *candidates = line
    for c in candidates:
        combined_text = f"{question} {c}"
        tokens = tokenizer.tokenize(combined_text)
        if tokens:
            corpus.append(tokens)

In [ ]:
len(corpus)

1256483

Для начала выберем window=5 и min_count=5

In [ ]:
from gensim.models import Word2Vec
embeddings_trained = Word2Vec(
    sentences=corpus,        # Корпус токенизированных текстов
    vector_size=200,         # Размерность векторов
    window=5,                # Размер окна контекста
    min_count=5,             # Минимальная частота слов
    workers=4                # Количество потоков
).wv

In [ ]:
wv_ranking = []
max_validation_examples = 1000
for i, line in enumerate(tqdm(validation_data)):
    if i == max_validation_examples:
        break
    q, *ex = line
    ranks = rank_candidates(q, ex, embeddings_trained, tokenizer)
    wv_ranking.append([r[0] for r in ranks].index(0) + 1)

  0%|          | 0/3760 [00:00<?, ?it/s]

In [ ]:
for k in tqdm([1, 5, 10, 100, 500, 1000]):
    print("DCG@%4d: %.3f | Hits@%4d: %.3f" % (k, dcg_score(wv_ranking, k), k, hits_count(wv_ranking, k)))

  0%|          | 0/6 [00:00<?, ?it/s]

DCG@   1: 0.264 | Hits@   1: 0.264
DCG@   5: 0.331 | Hits@   5: 0.395
DCG@  10: 0.354 | Hits@  10: 0.465
DCG@ 100: 0.406 | Hits@ 100: 0.723
DCG@ 500: 0.432 | Hits@ 500: 0.925
DCG@1000: 0.440 | Hits@1000: 1.000


Эксперимент с моим токенайзером

In [ ]:
tokenizer = ReallyMyTokenizer()

In [ ]:
# Создаем общий корпус текстов
corpus = []
for line in train_data:
    question, *candidates = line
    for c in candidates:
        combined_text = f"{question} {c}"
        tokens = tokenizer.tokenize(combined_text)
        if tokens:
            corpus.append(tokens)

In [ ]:
len(corpus)

1256483

In [ ]:
from gensim.models import Word2Vec
embeddings_trained = Word2Vec(
    sentences=corpus,        # Корпус токенизированных текстов
    vector_size=200,         # Размерность векторов
    window=5,                # Размер окна контекста
    min_count=5,             # Минимальная частота слов
    workers=4                # Количество потоков
).wv

In [ ]:
wv_ranking = []
max_validation_examples = 1000
for i, line in enumerate(tqdm(validation_data)):
    if i == max_validation_examples:
        break
    q, *ex = line
    ranks = rank_candidates(q, ex, embeddings_trained, tokenizer)
    wv_ranking.append([r[0] for r in ranks].index(0) + 1)

  0%|          | 0/3760 [00:00<?, ?it/s]

In [ ]:
for k in tqdm([1, 5, 10, 100, 500, 1000]):
    print("DCG@%4d: %.3f | Hits@%4d: %.3f" % (k, dcg_score(wv_ranking, k), k, hits_count(wv_ranking, k)))

  0%|          | 0/6 [00:00<?, ?it/s]

DCG@   1: 0.307 | Hits@   1: 0.307
DCG@   5: 0.380 | Hits@   5: 0.449
DCG@  10: 0.401 | Hits@  10: 0.515
DCG@ 100: 0.453 | Hits@ 100: 0.765
DCG@ 500: 0.475 | Hits@ 500: 0.941
DCG@1000: 0.482 | Hits@1000: 1.000


In [ ]:
import pandas as pd

rows = [
    {
        "tokenizer": "MyTokenizer",
        "window": 5, "min_count": 5,
        "DCG@1": 0.264, "Hits@1": 0.264,
        "DCG@5":  0.331, "Hits@5":  0.395,
        "DCG@10": 0.354, "Hits@10": 0.465,
        "DCG@100":0.406, "Hits@100":0.723,
        "DCG@500":0.432, "Hits@500":0.925,
        "DCG@1000":0.440, "Hits@1000":1.000,
    },
    {
        "tokenizer": "ReallyMyTokenizer",
        "window": 5, "min_count": 5,
        "DCG@1": 0.307, "Hits@1": 0.307,
        "DCG@5":  0.380, "Hits@5":  0.449,
        "DCG@10": 0.401, "Hits@10": 0.515,
        "DCG@100":0.453, "Hits@100":0.765,
        "DCG@500":0.475, "Hits@500":0.941,
        "DCG@1000":0.482, "Hits@1000":1.000,
    },
]

df_tok = pd.DataFrame(rows)
print(df_tok.to_string(index=False))

        tokenizer  window  min_count  DCG@1  Hits@1  DCG@5  Hits@5  DCG@10  Hits@10  DCG@100  Hits@100  DCG@500  Hits@500  DCG@1000  Hits@1000
      MyTokenizer       5          5  0.264   0.264  0.331   0.395   0.354    0.465    0.406     0.723    0.432     0.925     0.440        1.0
ReallyMyTokenizer       5          5  0.307   0.307  0.380   0.449   0.401    0.515    0.453     0.765    0.475     0.941     0.482        1.0


Можно заметить, что токенайзер, сохраняющий специальные символы (reallyMy), по результатам превосходит базовый. Потому дальнейшие эксперименты будем делать с тем, качество которого оказалось лучшим.

Увеличим размер окна

In [ ]:
embeddings_trained = Word2Vec(
    sentences=corpus,        # Корпус токенизированных текстов
    vector_size=200,         # Размерность векторов
    window=7,                # Размер окна контекста
    min_count=5,             # Минимальная частота слов
    workers=4                # Количество потоков
).wv

In [ ]:
wv_ranking = []
max_validation_examples = 1000
for i, line in enumerate(tqdm(validation_data)):
    if i == max_validation_examples:
        break
    q, *ex = line
    ranks = rank_candidates(q, ex, embeddings_trained, tokenizer)
    wv_ranking.append([r[0] for r in ranks].index(0) + 1)

  0%|          | 0/3760 [00:00<?, ?it/s]

In [ ]:
for k in tqdm([1, 5, 10, 100, 500, 1000]):
    print("DCG@%4d: %.3f | Hits@%4d: %.3f" % (k, dcg_score(wv_ranking, k), k, hits_count(wv_ranking, k)))

  0%|          | 0/6 [00:00<?, ?it/s]

DCG@   1: 0.311 | Hits@   1: 0.311
DCG@   5: 0.388 | Hits@   5: 0.458
DCG@  10: 0.407 | Hits@  10: 0.518
DCG@ 100: 0.461 | Hits@ 100: 0.775
DCG@ 500: 0.482 | Hits@ 500: 0.942
DCG@1000: 0.488 | Hits@1000: 1.000


In [ ]:
embeddings_trained = Word2Vec(
    sentences=corpus,        # Корпус токенизированных текстов
    vector_size=200,         # Размерность векторов
    window=10,                # Размер окна контекста
    min_count=5,             # Минимальная частота слов
    workers=4                # Количество потоков
).wv

In [ ]:
wv_ranking = []
max_validation_examples = 1000
for i, line in enumerate(tqdm(validation_data)):
    if i == max_validation_examples:
        break
    q, *ex = line
    ranks = rank_candidates(q, ex, embeddings_trained, tokenizer)
    wv_ranking.append([r[0] for r in ranks].index(0) + 1)

  0%|          | 0/3760 [00:00<?, ?it/s]

In [ ]:
for k in tqdm([1, 5, 10, 100, 500, 1000]):
    print("DCG@%4d: %.3f | Hits@%4d: %.3f" % (k, dcg_score(wv_ranking, k), k, hits_count(wv_ranking, k)))

  0%|          | 0/6 [00:00<?, ?it/s]

DCG@   1: 0.320 | Hits@   1: 0.320
DCG@   5: 0.399 | Hits@   5: 0.469
DCG@  10: 0.419 | Hits@  10: 0.532
DCG@ 100: 0.469 | Hits@ 100: 0.774
DCG@ 500: 0.491 | Hits@ 500: 0.943
DCG@1000: 0.497 | Hits@1000: 1.000


In [ ]:
embeddings_trained = Word2Vec(
    sentences=corpus,        # Корпус токенизированных текстов
    vector_size=200,         # Размерность векторов
    window=12,                # Размер окна контекста
    min_count=5,             # Минимальная частота слов
    workers=4                # Количество потоков
).wv

In [ ]:
wv_ranking = []
max_validation_examples = 1000
for i, line in enumerate(tqdm(validation_data)):
    if i == max_validation_examples:
        break
    q, *ex = line
    ranks = rank_candidates(q, ex, embeddings_trained, tokenizer)
    wv_ranking.append([r[0] for r in ranks].index(0) + 1)

  0%|          | 0/3760 [00:00<?, ?it/s]

In [ ]:
for k in tqdm([1, 5, 10, 100, 500, 1000]):
    print("DCG@%4d: %.3f | Hits@%4d: %.3f" % (k, dcg_score(wv_ranking, k), k, hits_count(wv_ranking, k)))

  0%|          | 0/6 [00:00<?, ?it/s]

DCG@   1: 0.322 | Hits@   1: 0.322
DCG@   5: 0.402 | Hits@   5: 0.473
DCG@  10: 0.421 | Hits@  10: 0.533
DCG@ 100: 0.473 | Hits@ 100: 0.781
DCG@ 500: 0.493 | Hits@ 500: 0.942
DCG@1000: 0.500 | Hits@1000: 1.000


Но в таком виде оно несмотрибельно, поэтому:

In [ ]:
import pandas as pd

results = [
    {"window": 5, "min_count": 5, "DCG@1": 0.307, "Hits@1": 0.307, "DCG@5": 0.380, "Hits@5": 0.449,
     "DCG@10": 0.401, "Hits@10": 0.515, "DCG@100": 0.453, "Hits@100": 0.765,
     "DCG@500": 0.475, "Hits@500": 0.941, "DCG@1000": 0.482, "Hits@1000": 1.000},

    {"window": 7, "min_count": 5, "DCG@1": 0.311, "Hits@1": 0.311, "DCG@5": 0.388, "Hits@5": 0.458,
     "DCG@10": 0.407, "Hits@10": 0.518, "DCG@100": 0.461, "Hits@100": 0.775,
     "DCG@500": 0.482, "Hits@500": 0.942, "DCG@1000": 0.488, "Hits@1000": 1.000},

    {"window": 10, "min_count": 5, "DCG@1": 0.320, "Hits@1": 0.320, "DCG@5": 0.399, "Hits@5": 0.469,
     "DCG@10": 0.419, "Hits@10": 0.532, "DCG@100": 0.469, "Hits@100": 0.774,
     "DCG@500": 0.491, "Hits@500": 0.943, "DCG@1000": 0.497, "Hits@1000": 1.000},

    {"window": 12, "min_count": 5, "DCG@1": 0.322, "Hits@1": 0.322, "DCG@5": 0.402, "Hits@5": 0.473,
     "DCG@10": 0.421, "Hits@10": 0.533, "DCG@100": 0.473, "Hits@100": 0.781,
     "DCG@500": 0.493, "Hits@500": 0.942, "DCG@1000": 0.500, "Hits@1000": 1.000},
]

df = pd.DataFrame(results)

print(df.to_string(index=False))


 window  min_count  DCG@1  Hits@1  DCG@5  Hits@5  DCG@10  Hits@10  DCG@100  Hits@100  DCG@500  Hits@500  DCG@1000  Hits@1000
      5          5  0.307   0.307  0.380   0.449   0.401    0.515    0.453     0.765    0.475     0.941     0.482        1.0
      7          5  0.311   0.311  0.388   0.458   0.407    0.518    0.461     0.775    0.482     0.942     0.488        1.0
     10          5  0.320   0.320  0.399   0.469   0.419    0.532    0.469     0.774    0.491     0.943     0.497        1.0
     12          5  0.322   0.322  0.402   0.473   0.421    0.533    0.473     0.781    0.493     0.942     0.500        1.0


Четко прослеживался тренд "чем больше, тем лучше".

Кроме Hits@100 для window = 10 и Hits@500 для window = 12.

Скорей всего, именно диапазон 10-12 является оптимальным: большой контекст, но без лишнего шума.

А теперь варьируем частоту слов:

In [ ]:
from gensim.models import Word2Vec
embeddings_trained = Word2Vec(
    sentences=corpus,        # Корпус токенизированных текстов
    vector_size=200,         # Размерность векторов
    window=12,                # Размер окна контекста
    min_count=3,             # Минимальная частота слов
    workers=4                # Количество потоков
).wv

In [ ]:
wv_ranking = []
max_validation_examples = 1000
for i, line in enumerate(tqdm(validation_data)):
    if i == max_validation_examples:
        break
    q, *ex = line
    ranks = rank_candidates(q, ex, embeddings_trained, tokenizer)
    wv_ranking.append([r[0] for r in ranks].index(0) + 1)

  0%|          | 0/3760 [00:00<?, ?it/s]

In [ ]:
for k in tqdm([1, 5, 10, 100, 500, 1000]):
    print("DCG@%4d: %.3f | Hits@%4d: %.3f" % (k, dcg_score(wv_ranking, k), k, hits_count(wv_ranking, k)))

  0%|          | 0/6 [00:00<?, ?it/s]

DCG@   1: 0.321 | Hits@   1: 0.321
DCG@   5: 0.401 | Hits@   5: 0.470
DCG@  10: 0.421 | Hits@  10: 0.532
DCG@ 100: 0.472 | Hits@ 100: 0.776
DCG@ 500: 0.494 | Hits@ 500: 0.943
DCG@1000: 0.500 | Hits@1000: 1.000


In [ ]:
from gensim.models import Word2Vec
embeddings_trained = Word2Vec(
    sentences=corpus,        # Корпус токенизированных текстов
    vector_size=200,         # Размерность векторов
    window=12,                # Размер окна контекста
    min_count=7,             # Минимальная частота слов
    workers=4                # Количество потоков
).wv

In [ ]:
wv_ranking = []
max_validation_examples = 1000
for i, line in enumerate(tqdm(validation_data)):
    if i == max_validation_examples:
        break
    q, *ex = line
    ranks = rank_candidates(q, ex, embeddings_trained, tokenizer)
    wv_ranking.append([r[0] for r in ranks].index(0) + 1)

  0%|          | 0/3760 [00:00<?, ?it/s]

In [ ]:
for k in tqdm([1, 5, 10, 100, 500, 1000]):
    print("DCG@%4d: %.3f | Hits@%4d: %.3f" % (k, dcg_score(wv_ranking, k), k, hits_count(wv_ranking, k)))


  0%|          | 0/6 [00:00<?, ?it/s]

DCG@   1: 0.323 | Hits@   1: 0.323
DCG@   5: 0.400 | Hits@   5: 0.465
DCG@  10: 0.420 | Hits@  10: 0.524
DCG@ 100: 0.474 | Hits@ 100: 0.783
DCG@ 500: 0.495 | Hits@ 500: 0.944
DCG@1000: 0.500 | Hits@1000: 1.000


Переведем в удобный для просмотра вид:

In [ ]:
import pandas as pd

results = [
    {"window": 12, "min_count": 3, "DCG@1": 0.321, "Hits@1": 0.321, "DCG@5": 0.401, "Hits@5": 0.470,
     "DCG@10": 0.421, "Hits@10": 0.532, "DCG@100": 0.472, "Hits@100": 0.776,
     "DCG@500": 0.494, "Hits@500": 0.943, "DCG@1000": 0.500, "Hits@1000": 1.000},

    {"window": 12, "min_count": 5, "DCG@1": 0.322, "Hits@1": 0.322, "DCG@5": 0.402, "Hits@5": 0.473,
     "DCG@10": 0.421, "Hits@10": 0.533, "DCG@100": 0.473, "Hits@100": 0.781,
     "DCG@500": 0.493, "Hits@500": 0.942, "DCG@1000": 0.500, "Hits@1000": 1.000},

    {"window": 12, "min_count": 7, "DCG@1": 0.323, "Hits@1": 0.323, "DCG@5": 0.400, "Hits@5": 0.465,
     "DCG@10": 0.420, "Hits@10": 0.524, "DCG@100": 0.474, "Hits@100": 0.783,
     "DCG@500": 0.495, "Hits@500": 0.944, "DCG@1000": 0.500, "Hits@1000": 1.000},
    ]

df = pd.DataFrame(results)

print(df.to_string(index=False))

 window  min_count  DCG@1  Hits@1  DCG@5  Hits@5  DCG@10  Hits@10  DCG@100  Hits@100  DCG@500  Hits@500  DCG@1000  Hits@1000
     12          3  0.321   0.321  0.401   0.470   0.421    0.532    0.472     0.776    0.494     0.943       0.5        1.0
     12          5  0.322   0.322  0.402   0.473   0.421    0.533    0.473     0.781    0.493     0.942       0.5        1.0
     12          7  0.323   0.323  0.400   0.465   0.420    0.524    0.474     0.783    0.495     0.944       0.5        1.0


При min_count=5 и min_count=3 результаты почти одинаковые, а при min_count=7 качество на @5 и @10 проседает.

Снижение порогового значения не отразилось, повышение - внесло дестабилизацию.

Возьмем как "золотой стандарт" window=12 и min_count=5.

А теперь добавим нормализацию: для каждого токена реализуем приведение к нижнему регистру, почистим токены с редкими символами, удалим стоп-слова,лемматизируем оставшееся и отбросим очень короткие токены.


In [ ]:
import re

USE_LEMMAS = False
try:
    import nltk
    from nltk.corpus import stopwords
    from nltk.stem import WordNetLemmatizer
    nltk.download('stopwords', quiet=True)
    nltk.download('wordnet', quiet=True)
    STOP = set(stopwords.words('english'))
    LEMMA = WordNetLemmatizer()
    USE_LEMMAS = True
except Exception:
    STOP = set()
TOKEN_RE = re.compile(r"[A-Za-z0-9_+#.-]+")

def normalize_tokens(raw_tokens):
    norm = []
    for t in raw_tokens:
        t = t.lower()
        if not TOKEN_RE.fullmatch(t):
            continue
        if t in STOP:
            continue
        if USE_LEMMAS:
            t = LEMMA.lemmatize(t)
        if len(t) < 2:
            continue
        norm.append(t)
    return norm


In [ ]:
train_data = read_corpus('/content/data/data/train.tsv')

corpus = []
for line in train_data:
    question, *candidates = line
    for c in candidates:
        combined_text = f"{question} {c}"
        raw = tokenizer.tokenize(combined_text)
        tokens = normalize_tokens(raw)
        if tokens:
            corpus.append(tokens)


In [ ]:
from gensim.models import Word2Vec
embeddings_trained = Word2Vec(
    sentences=corpus,        # Корпус токенизированных текстов
    vector_size=200,         # Размерность векторов
    window=12,                # Размер окна контекста
    min_count=5,             # Минимальная частота слов
    workers=4                # Количество потоков
).wv

In [ ]:
wv_ranking = []
max_validation_examples = 1000
for i, line in enumerate(tqdm(validation_data)):
    if i == max_validation_examples:
        break
    q, *ex = line
    ranks = rank_candidates(q, ex, embeddings_trained, tokenizer)
    wv_ranking.append([r[0] for r in ranks].index(0) + 1)

  0%|          | 0/3760 [00:00<?, ?it/s]

In [ ]:
for k in tqdm([1, 5, 10, 100, 500, 1000]):
    print("DCG@%4d: %.3f | Hits@%4d: %.3f" % (k, dcg_score(wv_ranking, k), k, hits_count(wv_ranking, k)))

  0%|          | 0/6 [00:00<?, ?it/s]

DCG@   1: 0.370 | Hits@   1: 0.370
DCG@   5: 0.463 | Hits@   5: 0.551
DCG@  10: 0.487 | Hits@  10: 0.624
DCG@ 100: 0.531 | Hits@ 100: 0.841
DCG@ 500: 0.546 | Hits@ 500: 0.958
DCG@1000: 0.551 | Hits@1000: 1.000


Можем заметить, что после нормализации метрики значительно улучшились, но все еще не годятся для демонстрации чудес в решении задачи ранжирования.

### Замечание:
Решить эту задачу с помощью обучения полноценной нейронной сети будет вам предложено, как часть задания в одной из домашних работ по теме "Диалоговые системы".

Напишите свой вывод о полученных результатах.
* Какой принцип токенизации даёт качество лучше и почему?
* Помогает ли нормализация слов?
* Какие эмбеддинги лучше справляются с задачей и почему?
* Почему получилось плохое качество решения задачи?
* Предложите свой подход к решению задачи.

## Вывод:


Простая токенизация через re.findall('\w+', text) часто не даёт оптимального результата: она не учитывает спецсимволы, сокращения и особенности корпуса. Поэтому лучше использовать более гибкие токенизаторы. Кроме того, результаты показывают, что лучше работает токенизация с учётом более широкого контекста (window больше). Тогда модель лучше улавливает смысловые связи.

Нормализация снижает количество искусственных различий между формами слов. Это особенно важно при разнообразных временах, склонениях, числах и т.д. Приведение к нижнему регистру, удаление пунктуации, лемматизация обычно повышает качество эмбеддингов и ускоряет обучение.

Эмбеддинги, обученные на корпусе похожих вопросов, лучше справляются с задачей, так как они учитывают контекст вопроса.

Плохое качество решения задачи, помимо не самого удачного выбора модели, может быть связано с недостаточным размером корпуса, а также с недостаточным размером или неправильным выбором окна при обучении эмбеддингов. Возможно также нейросеть воспринимает специальные наборы символов и сокращения как слова, что снижает качество восприятия.

Для улучшения качества решения задачи можно использовать более сложные модели и более детализированную предобработку.

Благодарю за прочтение 🙏 💙

Буду благодарна любой обратной связи :)